# Configure neutral sampling parameters

This notebook helps you configure unbiased ensemble generation parameters.

- No partisan metrics are computed.
- You can export a YAML config to run the full ensemble via `04_run_ensemble.py --config <yaml>`.


In [1]:
# Environment setup
import os
import json
from pathlib import Path

# Ensure results/configurations exists
Path("results/configurations").mkdir(parents=True, exist_ok=True)

In [2]:
from datetime import datetime

# User-defined tag for this configuration
config_tag = ""  # <-- Change this to something descriptive if desired

if config_tag:
    now_str = datetime.now().strftime("%Y%m%d%H%M%S")
    config_dir = os.path.join("results", "configurations", f"{now_str}_{config_tag}")
else:
    config_dir = os.path.join("results", "configurations", datetime.now().strftime("%Y%m%d%H%M%S"))
Path(config_dir).mkdir(parents=True, exist_ok=True)

# Configure parameters (edit these values)
config = {
    "steps": 51,
    "viz_every": 5,
    "use_cut_edges": True,
    "max_muni_splits": 5,
    "max_county_splits": 5,
    # Optional pre-optimization
    "enable_optimization": True,
    "optim_steps": 20,
    # Regional surcharges
    "muni_surcharge": 9.0,
    "county_surcharge": 3.0,
    "highered_surcharge": 1.0,
    "metro_surcharge": 1.0,
    "schdist_surcharge": 0.1,
    "water_surcharge": 0.1,
    "basin_surcharge": 0.1,
    "tilted_run": 0.5,
    # Optional fields supported by CLI; not used for neutral sampling inside this notebook
    "years": "2016,2020,2024",
    "offices": "PRE,GOV,ATG,AUD,TRE",
    "vote_share_agg": "median",
}

In [3]:
# Load data and build initial partition (no elections)
from utgc.data_io import load_data, load_county_boundaries, load_municipality_boundaries
from utgc.build import create_graph, create_updaters, create_initial_partition, create_constraints, create_proposal

precincts, initial_plan = load_data()
counties = load_county_boundaries(precincts)
municipalities = load_municipality_boundaries(precincts)

graph = create_graph(precincts)
# No elections for neutral sampling
updaters = create_updaters(elections=[], election_columns={})
initial_partition = create_initial_partition(graph, precincts, updaters)

ideal_population = sum(initial_partition["population"].values()) / len(initial_partition)
print(f"Ideal population per district: {ideal_population:,.0f}")

Loading data...
Loaded 2981 precincts
Loaded initial plan with 4 districts
Loading county boundaries from data/cois/UtahCountyBoundaries/ut_cnty_2020_bound.shp...
Loaded 29 counties
Loading municipality boundaries from data/cois/UtahMunicipalBoundaries/Municipalities.shp...
Loaded 259 municipalities
Creating graph...
Graph created with 2981 nodes and 8394 edges
Creating updaters...
Creating initial partition...
Initial partition created with 4 districts
Ideal population per district: 817,904


In [4]:
constraints = create_constraints(initial_partition,
                                 use_cut_edges=config["use_cut_edges"],
                                 max_muni_splits=config["max_muni_splits"],
                                 max_county_splits=config["max_county_splits"]) 

proposal = create_proposal(ideal_population,
                           precincts,
                           muni_surcharge=config["muni_surcharge"],
                           county_surcharge=config["county_surcharge"],
                           highered_surcharge=config["highered_surcharge"],
                           metro_surcharge=config["metro_surcharge"],
                           schdist_surcharge=config["schdist_surcharge"],
                           basin_surcharge=config["basin_surcharge"],
                           water_surcharge=config["water_surcharge"]) 

print("Configured parameters:")
for k, v in config.items():
    print(f"  {k}: {v}")

Creating constraints...
Adding cut edges constraint for compactness...
Creating ReCom proposal...
Region surcharges: {'MUNIID': 9.0, 'COUNTYID': 3.0, 'HIGHERED_ID': 1.0, 'METRO_ID': 1.0, 'SCHDIST_ID': 0.1, 'BASIN_ID': 0.1, 'WATER_ID': 0.1}
Configured parameters:
  steps: 51
  viz_every: 5
  use_cut_edges: True
  max_muni_splits: 5
  max_county_splits: 5
  enable_optimization: True
  optim_steps: 20
  muni_surcharge: 9.0
  county_surcharge: 3.0
  highered_surcharge: 1.0
  metro_surcharge: 1.0
  schdist_surcharge: 0.1
  water_surcharge: 0.1
  basin_surcharge: 0.1
  tilted_run: 0.5
  years: 2016,2020,2024
  offices: PRE,GOV,ATG,AUD,TRE
  vote_share_agg: median


In [5]:
# Optional pre-optimization step
from utgc.optimization import run_optimization

# Choose a starting partition based on whether optimization is enabled and constraints are satisfied
start_partition = initial_partition
if bool(config.get("enable_optimization", False)):
    print("Running pre-optimization...")
    optimized_partition = run_optimization(
        initial_partition,
        proposal,
        muni_surcharge=config["muni_surcharge"],
        county_surcharge=config["county_surcharge"],
        optimization_steps=int(config.get("optim_steps", 20)),
        split_munis_tolerance=config["max_muni_splits"],
        split_counties_tolerance=config["max_county_splits"],
    )
    
    def _satisfies_all(partition, constraints_list):
        for c in constraints_list:
            try:
                if not c(partition):
                    return False
            except Exception:
                return False
        return True

    if _satisfies_all(optimized_partition, constraints):
        print("Using optimized partition as starting point.")
        start_partition = optimized_partition
    elif _satisfies_all(initial_partition, constraints):
        print("Optimized partition failed constraints; using initial partition.")
        start_partition = initial_partition
    else:
        print("Neither optimized nor initial partition meets all constraints; proceeding with initial partition and filtered constraints.")



Running pre-optimization...
Running optimization for 20 steps...
Tolerance thresholds: pop_dev=0.0010, muni_splits=5, county_splits=5
Starting optimization...


100%|██████████| 20/20 [01:27<00:00,  4.38s/it]


Optimized score: 2.454623525499325
✗ Attempt 1 failed tolerance tests, retrying...
Retrying optimization (attempt 2/5)...


100%|██████████| 20/20 [00:28<00:00,  1.42s/it]

Optimized score: 1.7372503374479158
✓ Optimization successful on attempt 2! All tolerances met.
Final population deviation: 0.000737
Final municipality splits: 2
Final county splits: 3
Using optimized partition as starting point.


In [6]:
# Unique run tag and base directory for samples and YAML
# from datetime import datetime
# from pathlib import Path
# import os

# # Optional user tag (set UTGC_TAG env var or edit here)
# user_tag = os.environ.get("UTGC_TAG", "").strip()
# run_tag = f"{datetime.now().strftime('%Y%m%d-%H%M%S')}{('-' + user_tag) if user_tag else ''}"

# sample_dir = Path("results") / "configurations" / run_tag
# sample_dir.mkdir(parents=True, exist_ok=True)

# # include tag in config for downstream consumers
# config["tag"] = run_tag

# print(f"Run tag: {run_tag}")
# print(f"Sample outputs and YAML will be saved under: {sample_dir}")


In [7]:
# Run a short neutral sample and visualize into the tag directory
from utgc.ensemble import run_ensemble
from utgc.reporting import save_visualization

# Wrap visualization to write into the tag-specific directory

def _viz(partition, step, res, counties, municipalities):
    save_visualization(partition, step, res, counties, municipalities, base_dir=str(config_dir))

results = run_ensemble(
    start_partition,
    proposal,
    constraints,
    available_elections=[],
    counties=counties,
    municipalities=municipalities,
    num_steps=int(config["steps"]),
    visualize_every=int(config["viz_every"]),
    vote_share_agg=str(config["vote_share_agg"]),
    save_visualization_fn=_viz,
)

print(f"Completed {len(results)} steps.")
print(f"Maps saved to: {config_dir}")


Running ensemble analysis with 51 steps...


  0%|          | 0/51 [00:00<?, ?it/s]

Completed 51 steps.
Maps saved to: results/configurations/20251007124254


In [8]:
# Show a small neutral metrics preview
def _summarize(results):
    rows = []
    for r in results[: min(5, len(results))]:
        rows.append({
            "step": r.get("step"),
            "split_munis_count": r.get("split_munis_count"),
            "split_counties_count": r.get("split_counties_count"),
        })
    return rows

preview = _summarize(results)
preview


[{'step': 0, 'split_munis_count': 2, 'split_counties_count': 3},
 {'step': 1, 'split_munis_count': 2, 'split_counties_count': 3},
 {'step': 2, 'split_munis_count': 2, 'split_counties_count': 3},
 {'step': 3, 'split_munis_count': 2, 'split_counties_count': 3},
 {'step': 4, 'split_munis_count': 2, 'split_counties_count': 3}]

In [9]:
# Export YAML configuration for CLI runs inside the tag folder
import yaml

yaml_path = os.path.join(config_dir, "params.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=True)

print(f"Wrote YAML: {yaml_path}")


Wrote YAML: results/configurations/20251007124254/params.yaml
